# Pipeline de Ingestão RAG — Passo a Passo

Este notebook explica e demonstra cada etapa do pipeline de ingestão de documentos para o sistema RAG do Compliance Viewer.

O objetivo é entender, de forma didática, o caminho completo que um documento percorre até virar um chunk pesquisável no banco vetorial:

> **LÊ → EXTRAI → CORTA → EMBEDA → JOGA NO CHROMADB**

Cada seção abaixo isola uma etapa, mostra o código e o resultado intermediário.

## 0. Setup

Importações e configurações. O cliente Azure OpenAI é usado para gerar os embeddings (`text-embedding-ada-002`, 1536 dimensões).

> Rode este notebook a partir da **raiz do projeto** (onde fica a pasta `knowledge_base/`).

In [23]:
import os
import time
from pathlib import Path

import chromadb
from dotenv import load_dotenv
from openai import AzureOpenAI
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# O notebook está em notebooks/, então subimos um nível para a raiz do projeto.
# Assim os caminhos relativos funcionam igual ao código em src/.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

print(f"Working directory: {Path.cwd()}")

# Carrega as credenciais do .env (que está na raiz)
load_dotenv()

# Configurações
KNOWLEDGE_BASE_DIR = "knowledge_base"
CHROMA_DB_PATH = "data/chroma_db"
COLLECTION_NAME = "compliance_docs"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
EMBEDDING_MODEL = "text-embedding-ada-002"

print("Setup concluído.")

Working directory: c:\Users\BB442HD\OneDrive - EY\Desktop\compliance-viewer
Setup concluído.


## 1. LER — Quais documentos existem na knowledge base?

A primeira etapa e descobrir quais arquivos vamos processar. O pipeline aceita `.pdf` e `.txt`.

A knowledge base contem documentos normativos oficiais: codigos da ANBIMA, resolucoes da CVM e politicas internas de adequacao de perfil.

In [24]:
files = [f for f in os.listdir(KNOWLEDGE_BASE_DIR) if f.endswith((".pdf", ".txt"))]

print(f"Documentos encontrados ({len(files)}):")
for f in files:
    print(f"  - {f}")

Documentos encontrados (6):
  - anbima_codigo_distribuicao_produtos_Investimento.pdf
  - email_analise_cliente_01.txt
  - manual_comunicacao_cliente_v1.0.txt
  - politica_adequacao_investimento_v1.2.txt
  - politica_investimento_agressivo_v1.0.txt
  - resol_030_cvm.pdf


## 2. EXTRAIR — Tirar o texto bruto de cada documento

PDFs e TXTs precisam de tratamento diferente:
- **PDF**: usamos o `pypdf` para extrair o texto pagina por pagina.
- **TXT**: leitura direta do arquivo.

Vamos extrair o texto de um documento de exemplo e ver os primeiros caracteres.

In [19]:
def extract_text_from_pdf(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text

def extract_text_from_txt(txt_path: str) -> str:
    with open(txt_path, "r", encoding="utf-8") as f:
        return f.read()

# Exemplo: extrai o primeiro arquivo da lista
exemplo = files[0]
caminho = os.path.join(KNOWLEDGE_BASE_DIR, exemplo)
texto = extract_text_from_pdf(caminho) if exemplo.endswith(".pdf") else extract_text_from_txt(caminho)

print(f"Documento: {exemplo}")
print(f"Tamanho total: {len(texto)} caracteres\n")
print("Primeiros 500 caracteres:")
print(texto[:500])

Documento: anbima_codigo_distribuicao_produtos_Investimento.pdf
Tamanho total: 115476 caracteres

Primeiros 500 caracteres:
 
 
  
Código de 
Distribuição de Produtos  
de Investimento 
 
2 
 
Sumário 
CAPÍTULO I – DEFINIÇÕES ........................................................................................................ 4 
CAPÍTULO II – OBJETIVO E ABRANGÊNCIA .............................................................................. 8 
CAPÍTULO III – ASSOCIAÇÃO E ADESÃO ANBIMA .................................................................. 10 
CAPÍTULO IV - PRINCÍPIOS GERAIS DE CONDUTA ...............


## 3. CORTAR — Dividir o texto em chunks

Um documento inteiro e grande demais para ser util na busca. Dividimos em **chunks** (pedacos menores).

Usamos o `RecursiveCharacterTextSplitter` do LangChain, que e mais inteligente que um corte cego por tamanho: ele tenta quebrar primeiro em paragrafos, depois em frases, depois em palavras. Isso preserva o significado.

- **chunk_size = 500**: cada chunk tem ~500 caracteres
- **chunk_overlap = 50**: cada chunk compartilha 50 caracteres com o anterior, para nao perder contexto nas bordas

In [20]:
def split_into_chunks(text: str) -> list[str]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_text(text)
    return [c.strip() for c in chunks if c.strip()]

chunks = split_into_chunks(texto)

print(f"O documento foi dividido em {len(chunks)} chunks.\n")
print("=== Chunk 0 ===")
print(chunks[0])
print("\n=== Chunk 1 ===")
print(chunks[1])

O documento foi dividido em 258 chunks.

=== Chunk 0 ===
Código de 
Distribuição de Produtos  
de Investimento 
 
2 
 
Sumário 
CAPÍTULO I – DEFINIÇÕES ........................................................................................................ 4 
CAPÍTULO II – OBJETIVO E ABRANGÊNCIA .............................................................................. 8 
CAPÍTULO III – ASSOCIAÇÃO E ADESÃO ANBIMA .................................................................. 10

=== Chunk 1 ===
CAPÍTULO IV - PRINCÍPIOS GERAIS DE CONDUTA ................................................................. 11 
CAPÍTULO V – REGRAS E PROCEDIMENTOS .......................................................................... 12 
SEÇÃO I – CONTROLES INTERNOS E/OU COMPLIANCE ............................................... 12 
SEÇÃO II – PRIVACIDADE E PROTEÇÃO DE DADOS PESSOAIS ...................................... 14


### Visualizando o overlap

Repare que o final de um chunk se repete no inicio do proximo. Esse e o **overlap** — ele garante que uma frase cortada na fronteira nao perca o contexto.

In [21]:
print("Final do Chunk 0:")
print(f"  ...{chunks[0][-80:]}")
print("\nInicio do Chunk 1:")
print(f"  {chunks[1][:80]}...")

Final do Chunk 0:
  ...SÃO ANBIMA .................................................................. 10

Inicio do Chunk 1:
  CAPÍTULO IV - PRINCÍPIOS GERAIS DE CONDUTA ........................................


## 4. EMBEDAR — Transformar cada chunk em um vetor

Aqui esta o coracao do RAG. Cada chunk de texto e convertido em um **embedding**: um vetor de 1536 numeros que representa o *significado* do texto.

Textos com significado parecido geram vetores proximos no espaco vetorial. E isso que permite a busca semantica depois.

Usamos o `text-embedding-ada-002` do Azure OpenAI. Vamos embedar um unico chunk para ver o resultado.

In [22]:
client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
)

def get_embedding(text: str) -> list:
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=text)
    return response.data[0].embedding

# Embeda o primeiro chunk
embedding_exemplo = get_embedding(chunks[0])

print(f"Chunk embedado: {chunks[0][:60]}...")
print(f"\nDimensoes do vetor: {len(embedding_exemplo)}")
print("Primeiros 10 valores do vetor:")
print(embedding_exemplo[:10])

Chunk embedado: Código de 
Distribuição de Produtos  
de Investimento 
 
2 
...

Dimensoes do vetor: 1536
Primeiros 10 valores do vetor:
[-0.021886825561523438, -0.017255662009119987, 0.021062634885311127, -0.02205689623951912, -0.03772959113121033, 0.014351372607052326, -0.0016369330696761608, 0.0026671707164496183, -0.005491329822689295, -0.017648132517933846]


### Por que isso importa?

Esse vetor de 1536 numeros e o que torna a busca semantica possivel. Quando o usuario faz uma pergunta, geramos o embedding da pergunta e procuramos os chunks cujos vetores estao mais proximos — ou seja, os mais parecidos em significado.

> **Importante:** na ingestao real geramos os embeddings em *batch* (varios chunks por chamada) para nao estourar o rate limit do Azure. Aqui fizemos um por vez so para fins didaticos.

## 5. JOGAR NO CHROMADB — Armazenar tudo no banco vetorial

Por fim, guardamos cada chunk no **ChromaDB** junto com:
- O **texto** do chunk (`documents`)
- O **embedding** (`embeddings`)
- Os **metadados** para rastreabilidade (`source`, `chunk_index`, `chunk_id`)

O `chunk_id` e o **rotulo** de cada chunk — permite saber exatamente de qual documento e posicao cada pedaco veio. Isso e essencial para a auditabilidade do compliance.

In [25]:
# Conecta ao ChromaDB
chroma_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

# Recria a collection do zero para o demo
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = chroma_client.create_collection(name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"})

# Joga os primeiros 5 chunks do documento de exemplo no banco
embeddings = [get_embedding(c) for c in chunks[:5]]
ids = [f"{exemplo}_chunk_{i}" for i in range(5)]
metadatas = [
    {"source": exemplo, "chunk_index": i, "chunk_id": f"{exemplo}_chunk_{i}"}
    for i in range(5)
]

collection.add(
    ids=ids,
    documents=chunks[:5],
    embeddings=embeddings,
    metadatas=metadatas,
)

print("5 chunks armazenados no ChromaDB.")
print(f"Total de chunks na collection: {collection.count()}")

5 chunks armazenados no ChromaDB.
Total de chunks na collection: 5


### Conferindo o que foi armazenado

Vamos inspecionar um chunk salvo, com seu rotulo (`chunk_id`) e metadados.

In [26]:
resultado = collection.get(ids=[f"{exemplo}_chunk_0"], include=["documents", "metadatas"])

print("Chunk recuperado do banco:")
print(f"  ID:     {resultado['ids'][0]}")
print(f"  Source: {resultado['metadatas'][0]['source']}")
print(f"  Indice: {resultado['metadatas'][0]['chunk_index']}")
print(f"  Texto:  {resultado['documents'][0][:150]}...")

Chunk recuperado do banco:
  ID:     anbima_codigo_distribuicao_produtos_Investimento.pdf_chunk_0
  Source: anbima_codigo_distribuicao_produtos_Investimento.pdf
  Indice: 0
  Texto:  Código de 
Distribuição de Produtos  
de Investimento 
 
2 
 
Sumário 
CAPÍTULO I – DEFINIÇÕES ..........................................................


## Resumo do Pipeline

Percorremos as 5 etapas da ingestao:

| Etapa | O que faz | Ferramenta |
|---|---|---|
| **1. LER** | Lista os documentos da knowledge base | `os.listdir` |
| **2. EXTRAIR** | Tira o texto bruto de PDFs e TXTs | `pypdf` |
| **3. CORTAR** | Divide em chunks com overlap | LangChain `RecursiveCharacterTextSplitter` |
| **4. EMBEDAR** | Converte cada chunk em vetor de 1536 dims | Azure `text-embedding-ada-002` |
| **5. JOGAR** | Armazena texto + vetor + metadados | ChromaDB |

O script de producao (`src/rag/ingestion.py`) faz exatamente isso, mas processando **todos** os documentos e em **batch** para eficiencia.

Depois da ingestao, o banco vetorial esta pronto para o **retrieval**: dada uma pergunta, encontrar os chunks mais relevantes por similaridade semantica.

## 6. RECUPERAR — Busca semantica com score de similaridade

Com os chunks no banco, podemos fazer a **recuperacao**: dada uma pergunta,
geramos o embedding dela e buscamos os chunks cujos vetores estao mais proximos.

Cada chunk retornado vem com seu **score de similaridade** (0 a 1). Quanto mais
perto de 1, mais o significado do chunk se aproxima da pergunta. Esse score e o
que torna a recuperacao **auditavel** — da pra justificar por que cada trecho foi
escolhido para embasar a resposta do agente.

In [27]:
pergunta = "Quais sao as regras para um perfil conservador?"

# Gera o embedding da pergunta (mesmo modelo da ingestao)
query_embedding = get_embedding(pergunta)

# Busca os chunks mais proximos no ChromaDB
resultados = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=["documents", "metadatas", "distances"],
)

print(f"Pergunta: {pergunta}\n")
print(f"Top {len(resultados['ids'][0])} chunks mais relevantes:\n")

for i in range(len(resultados["ids"][0])):
    chunk_id = resultados["ids"][0][i]
    texto_chunk = resultados["documents"][0][i]
    distancia = resultados["distances"][0][i]
    similaridade = 1 - distancia  # com hnsw:space=cosine, distancia = 1 - similaridade

    print(f"--- Resultado {i + 1} ---")
    print(f"  chunk_id:      {chunk_id}")
    print(f"  similaridade:  {similaridade:.4f}")
    print(f"  texto:         {texto_chunk[:150]}...")
    print()

Pergunta: Quais sao as regras para um perfil conservador?

Top 3 chunks mais relevantes:

--- Resultado 1 ---
  chunk_id:      anbima_codigo_distribuicao_produtos_Investimento.pdf_chunk_1
  similaridade:  0.7763
  texto:         CAPÍTULO IV - PRINCÍPIOS GERAIS DE CONDUTA ................................................................. 11 
CAPÍTULO V – REGRAS E PROCEDIMENTOS ....

--- Resultado 2 ---
  chunk_id:      anbima_codigo_distribuicao_produtos_Investimento.pdf_chunk_4
  similaridade:  0.7474
  texto:         SEÇÃO III – ASSESSOR DE INVESTIMENTOS ................................................................. 26 
CAPÍTULO VIII – PUBLICIDADE .................

--- Resultado 3 ---
  chunk_id:      anbima_codigo_distribuicao_produtos_Investimento.pdf_chunk_3
  similaridade:  0.7470
  texto:         CAPÍTULO VI – QUALIFICAÇÃO E TREINAMENTO .................................................................. 21 
CAPÍTULO VII – CONTRATAÇÃO DE TERCEIRO...

